# 04 — RL Training with Gymnasium

Train a reinforcement learning agent using the standard Gymnasium interface.

**What you'll learn**:
- Create a 3we Gymnasium environment
- Understand observation and action spaces
- Define a custom reward function
- Train a PPO agent with stable-baselines3

**Prerequisites**: `pip install threewe[sim] stable-baselines3`

In [ ]:
import gymnasium
import numpy as np
import threewe.gym  # registers 3we environments

## Create the Environment

3we provides pre-built Gymnasium environments:
- `3we/Navigation-v1` — Point-to-point navigation
- `3we/Exploration-v1` — Coverage exploration
- `3we/ObjectNav-v1` — Find a target object
- `3we/VLN-v1` — Vision-language navigation

In [ ]:
env = gymnasium.make("3we/Navigation-v1")

print("Observation space:")
for key, space in env.observation_space.spaces.items():
    print(f"  {key}: shape={space.shape}, dtype={space.dtype}")

print(f"\nAction space: {env.action_space}")
print(f"  Shape: {env.action_space.shape}")
print(f"  Range: [{env.action_space.low[0]}, {env.action_space.high[0]}]")

## Manual Environment Interaction

Standard Gymnasium loop — nothing 3we-specific here.

In [ ]:
obs, info = env.reset(seed=42)
print(f"Initial observation keys: {list(obs.keys())}")
print(f"Goal position: {obs['goal']}")

total_reward = 0.0
for step in range(20):
    action = env.action_space.sample()  # random policy
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    if terminated or truncated:
        print(f"  Episode ended at step {step+1}: reward={total_reward:.2f}")
        break

print(f"Final reward: {total_reward:.2f}")
env.close()

## Train a PPO Agent

Using stable-baselines3 for a simple training loop.
The environment is fully Gymnasium-compatible — any RL library works.

In [ ]:
try:
    from stable_baselines3 import PPO
    from stable_baselines3.common.vec_env import DummyVecEnv

    # Create vectorized environment
    vec_env = DummyVecEnv([lambda: gymnasium.make("3we/Navigation-v1")])

    # Initialize PPO with MultiInputPolicy (handles Dict observation spaces)
    model = PPO(
        "MultiInputPolicy",
        vec_env,
        learning_rate=3e-4,
        n_steps=256,
        batch_size=64,
        n_epochs=10,
        verbose=1,
    )

    print("Training PPO for 5000 steps (quick demo)...")
    model.learn(total_timesteps=5000)
    print("Training complete!")

    vec_env.close()

except ImportError:
    print("Install stable-baselines3 for training: pip install stable-baselines3")
    print("Skipping training demo.")

## Evaluate the Trained Agent

In [ ]:
try:
    eval_env = gymnasium.make("3we/Navigation-v1")
    successes = 0
    num_eval_episodes = 10

    for ep in range(num_eval_episodes):
        obs, info = eval_env.reset()
        done = False
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = eval_env.step(action)
            done = terminated or truncated
        if info.get("success", False):
            successes += 1

    print(f"Evaluation: {successes}/{num_eval_episodes} episodes successful")
    print(f"Success rate: {successes/num_eval_episodes:.0%}")
    eval_env.close()

except NameError:
    print("Model not trained — skipping evaluation.")

## Deploy to Real Robot

The trained policy works on real hardware with zero code changes:

```python
from threewe import Robot

async with Robot(backend="real") as robot:
    obs = robot.get_observation()
    action, _ = model.predict(obs, deterministic=True)
    robot.execute_action(action)
```

The observation format is identical between sim and real backends.